<a href="https://colab.research.google.com/github/tzelleke/jupyter-notebooks/blob/master/pandas/string-processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install nb2hugo

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import io
import pandas as pd

## Phone numbers - simple

Let us create some sample data.

In [ ]:
csv = io.StringIO('\n'.join([
    'ID,Name,Phones',
    '1,Russel,"01234567\n02345678\n03456789"',
    '2,Patrick,"04123456\n05234567\n06345678"',
]))

df = pd.read_csv(csv, index_col='ID')
df

,Name,Phones
ID,,
1,Russel,01234567\n02345678\n03456789
2,Patrick,04123456\n05234567\n06345678


The `Phones` column in this dataframe contains multiple phone numbers separated by line breaks for each record.
We are going to reshape the dataframe to hold

In [ ]:
df.assign(
    Phones=df.Phones.str.split('\n')
).explode('Phones')

,Name,Phones
ID,,
1,Russel,01234567
1,Russel,02345678
1,Russel,03456789
2,Patrick,04123456
2,Patrick,05234567
2,Patrick,06345678


Note, that the index `ID` is no longer unique.
`DataFrame.assign` is used to add new columns to a dataframe.


## Phone numbers with phone type

Next, we are going to look at a more complicated case where

In [ ]:
csv = io.StringIO('\n'.join([
    'ID,Name,Phones',
    '1,Russel,"mobil: 01234567\nwork: 02345678\nprivate: 03456789"',
    '2,Patrick,"mobil: 04123456\nprivate: 05234567"',
]))

df = pd.read_csv(csv, index_col='ID')
df

,Name,Phones
ID,,
1,Russel,mobil: 01234567\nwork: 02345678\nprivate: 0345...
2,Patrick,mobil: 04123456\nprivate: 05234567


dfdsfadsgfasdfa

In [ ]:
df.join(
    df.Phones
        .str.split('\n').explode()
        .str.split(':', expand=True)
        .rename(columns={
            0: 'phone_type',
            1: 'phone_number',
        })
).drop('Phones', axis='columns')

,Name,phone_type,phone_number
ID,,,
1,Russel,mobil,01234567
1,Russel,work,02345678
1,Russel,private,03456789
2,Patrick,mobil,04123456
2,Patrick,private,05234567


### Reshaping to wide format

We can reshape this dataframe to a wide format by using the idiom  
`DataFrame.set_index('<COLUMN>', append=True).unstack('<COLUMN>')`.  
[set_index](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.set_index.html) sets the DataFrame index using existing columns.
If we specify `append=True` the column is added to the existing index instead of replacing it. Thus, after this operation the dataframe has a MultiIndex.

In [ ]:
(
    df.Phones
        .str.split('\n').explode()
        .str.split(':', expand=True)
        .rename(columns={
            0: 'phone_type',
            1: 'phone_number',
        })
        .set_index('phone_type', append=True)
        .unstack('phone_type').droplevel(0, axis='columns')
)

phone_type,mobil,private,work
ID,,,
1,01234567,03456789,02345678
2,04123456,05234567,NaN


gfafgasdgfasdf

In [ ]:
df.join(
    df.Phones
        .str.split('\n').explode()
        .str.split(':', expand=True)
        .rename(columns={
            0: 'phone_type',
            1: 'phone_number',
        })
        .set_index(['phone_type'], append=True)
        .unstack('phone_type').droplevel(0, axis='columns')
).drop('Phones', axis='columns')

,Name,mobil,private,work
ID,,,,
1,Russel,01234567,03456789,02345678
2,Patrick,04123456,05234567,NaN


In [ ]:
import re

from traitlets import List
from nbconvert.preprocessors import Preprocessor
from nb2hugo.exporter import HugoExporter
from nb2hugo.writer import HugoWriter

In [ ]:
class TableExtractor(Preprocessor):
    pattern_table = re.compile(r'(?s:<table\b.*</table>\n)')
    pattern_table_border = re.compile(r'(?<=<table\b)(\sborder=\S+)')
    pattern_table_style = re.compile(r'(?<=\w\b)(\sstyle=[^>]+)')

    def preprocess_cell(self, cell, resources, index):
        if cell.cell_type == 'code' and len(cell.outputs):
            html = cell.outputs[0]['data']['text/html']
            html = TableExtractor.pattern_table.search(html).group(0)
            html = TableExtractor.pattern_table_border.sub('', html)
            html = TableExtractor.pattern_table_style.sub('', html)
            cell.outputs[0]['data']['text/html'] = html
        return cell, resources

In [ ]:
class Exporter(HugoExporter):
    preprocessors = List([TableExtractor]).tag(config=True)

In [ ]:
class Writer(HugoWriter):
    def __init__(self):
        self._exporter = Exporter()

In [ ]:
w = Writer()
w.convert('pandas.ipynb', '.', 'demo')

FileNotFoundError: ignored